In [1]:
from DWG_to_DuckDB_ETL import *
import os
import shutil

dwg_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\1 - Reask Solar Farm\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout"
temp_folder = r"C:\Users\Usuario\Ecotechnee\Office - Documentos\Projetos\422 - Zagope - UFV Reask e Tooletown\1 - Reask Solar Farm\01 - Projetos\01 - Em Confecção\02 - Edição\01 - Layout\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Reask - BOQ.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Reask - BOQ.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Reask - BOQ.dwg")

conn = load_to_duckdb(dxf_file, "PV Reask")
print("Data Loaded")
print("DuckDB Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Project PV Reask already exists
Removed existing data for project: PV Reask
Successfully loaded project
Loaded 19744 entities for project: PV Reask
Don't forget to close connection.
Data Loaded
DuckDB Loaded


CABLE AND STRUCTURE DATA
=

In [2]:
data = conn.sql("SELECT name, layer, X, Y, start_x, start_y, end_x, end_y, length FROM cad_entities WHERE project_name='PV Reask'")
print(data)
print(conn.execute("DESCRIBE cad_entities").fetchdf())

result = conn.execute("""
    SELECT
        layer,
        COUNT(*) as count
    FROM cad_entities
    GROUP BY layer
    ORDER BY count DESC
""").fetchdf()
print(result)

┌─────────┬─────────────────────────┬───────┬───────┬───────────┬──────────┬───────────┬───────────┬───────────┐
│  name   │          layer          │   X   │   Y   │  start_x  │ start_y  │   end_x   │   end_y   │  length   │
│ varchar │         varchar         │ float │ float │   float   │  float   │   float   │   float   │   float   │
├─────────┼─────────────────────────┼───────┼───────┼───────────┼──────────┼───────────┼───────────┼───────────┤
│ NULL    │ EMF_DC_vala             │  NULL │  NULL │  700579.6 │ 756157.0 │  700591.4 │  756157.0 │     11.75 │
│ NULL    │ EMF_DC_vala             │  NULL │  NULL │  700365.8 │ 756686.0 │  700365.8 │  756675.0 │      11.0 │
│ NULL    │ EMF_DC_vala             │  NULL │  NULL │  700401.6 │ 756675.0 │ 700402.75 │  756675.0 │  1.098843 │
│ NULL    │ EMF_DC_vala             │  NULL │  NULL │ 700402.75 │ 756675.0 │ 700436.25 │  756667.4 │ 36.069645 │
│ NULL    │ EMF_DC_vala             │  NULL │  NULL │ 700437.44 │ 756664.0 │ 700436.25 │  756664

STRUCTURE DATA

In [3]:
structures = conn.sql("SELECT layer, X, Y FROM data WHERE name = 'EMF422-1_string_tilt_15°' or name = '*U2'")
print(structures)

┌────────────────┬───────────┬───────────┐
│     layer      │     X     │     Y     │
│    varchar     │   float   │   float   │
├────────────────┼───────────┼───────────┤
│ EMF_em_pst_3.3 │ 699739.44 │ 755526.94 │
│ EMF_em_pst_3.3 │  699751.2 │ 755526.94 │
│ EMF_em_pst_3.3 │  699762.9 │ 755526.94 │
│ EMF_em_pst_3.3 │  699774.6 │ 755526.94 │
│ EMF_em_pst_3.3 │  699786.3 │ 755526.94 │
│ EMF_em_pst_3.3 │ 699798.06 │ 755526.94 │
│ EMF_em_pst_3.3 │ 699809.75 │ 755526.94 │
│ EMF_em_pst_3.3 │  699821.5 │ 755526.94 │
│ EMF_em_pst_3.3 │ 699833.25 │ 755526.94 │
│ EMF_em_pst_3.3 │ 699844.94 │ 755526.94 │
│       ·        │      ·    │     ·     │
│       ·        │      ·    │     ·     │
│       ·        │      ·    │     ·     │
│ EMF_em_pst_2.2 │  700633.0 │ 755940.44 │
│ EMF_em_pst_2.2 │ 700726.75 │ 755908.94 │
│ EMF_em_pst_3.3 │ 700151.75 │  755207.6 │
│ EMF_em_pst_1.3 │  700682.9 │  756291.5 │
│ EMF_em_pst_1.3 │ 700718.06 │  756281.0 │
│ EMF_em_pst_1.3 │ 700800.06 │  756260.0 │
│ EMF_em_ps

In [4]:
structure_df = structures.fetchdf()
structure_df['Line'] = round(structure_df['Y'],0)

structure_df.sort_values(by=["Line", "X"], ascending=[False,True], inplace=True, ignore_index=True)

#Different Lines
condition1 = structure_df["Line"] != structure_df["Line"].shift()

#Break in the Structure with no C type on the edges
condition2 = (abs(structure_df["X"] - structure_df["X"].shift()) > 13)


break_condition = condition1 | condition2

structure_df['Group'] = break_condition.cumsum()
print(f'The number of structures is: {max(structure_df['Group'])}')

The number of structures is: 452


CABLE DATA

In [5]:
ac_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_vala_btca_%xC'
""")

ac_access = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_access_ductbank_%xC'
""")

dc_cables = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer = 'EMF_cabo_solar_negativo' OR layer = 'EMF_cabo_solar_positivo'
""")

dc_ducts = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer LIKE 'EMF_bt_eletroduto_%S'
""")

dc_trenches = conn.sql("""
    SELECT layer, start_x, start_y, end_x, end_y, length FROM data WHERE layer = 'EMF_DC_vala'
""")

QTY. OF AC CABLES (m)
=

In [6]:
print(ac_cables)

┌───────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│         layer         │  start_x  │  start_y  │   end_x   │   end_y   │  length   │
│        varchar        │   float   │   float   │   float   │   float   │   float   │
├───────────────────────┼───────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_bt_vala_btca_01xC │  700435.8 │ 756667.75 │  700435.8 │ 756668.75 │       1.0 │
│ EMF_bt_vala_btca_01xC │  700519.8 │ 756634.75 │  700519.8 │ 756636.75 │       2.0 │
│ EMF_bt_vala_btca_01xC │  700594.9 │ 756601.75 │  700594.9 │ 756603.75 │       2.0 │
│ EMF_bt_vala_btca_01xC │ 700597.56 │ 756568.75 │ 700597.56 │ 756569.75 │       1.0 │
│ EMF_bt_vala_btca_01xC │ 700597.56 │ 756535.75 │ 700597.56 │ 756536.75 │       1.0 │
│ EMF_bt_vala_btca_01xC │ 700597.56 │ 756524.75 │ 700597.56 │ 756525.75 │       1.0 │
│ EMF_bt_vala_btca_01xC │  700595.6 │ 756491.75 │  700595.6 │ 756492.75 │       1.0 │
│ EMF_bt_vala_btca_01xC │ 700599.56 │ 756458.75 │ 7005

In [7]:
ac_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_cables' GROUP BY layer ORDER BY layer")
print(ac_cables_group)

┌───────────────────────┬────────────────────┐
│         layer         │    total_length    │
│        varchar        │       double       │
├───────────────────────┼────────────────────┤
│ EMF_bt_vala_btca_01xC │   2949.05357336998 │
│ EMF_bt_vala_btca_02xC │ 1078.4604984521866 │
│ EMF_bt_vala_btca_03xC │   962.562077999115 │
│ EMF_bt_vala_btca_04xC │  834.3671464920044 │
│ EMF_bt_vala_btca_05xC │ 1210.1421482563019 │
│ EMF_bt_vala_btca_06xC │ 171.76502656936646 │
│ EMF_bt_vala_btca_07xC │  514.0867881774902 │
│ EMF_bt_vala_btca_08xC │ 128.59579277038574 │
│ EMF_bt_vala_btca_09xC │ 416.77041935920715 │
│ EMF_bt_vala_btca_10xC │   303.166974067688 │
└───────────────────────┴────────────────────┘
  10 rows                          2 columns



QTY. OF AC DUCTS IN ACCESS (m and un.)
=

In [8]:
print(ac_access)

┌─────────────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│            layer            │  start_x  │  start_y  │   end_x   │   end_y   │  length   │
│           varchar           │   float   │   float   │   float   │   float   │   float   │
├─────────────────────────────┼───────────┼───────────┼───────────┼───────────┼───────────┤
│ EMF_bt_access_ductbank_05xC │  700150.2 │  755699.3 │  700156.1 │ 755700.06 │  6.000982 │
│ EMF_bt_access_ductbank_04xC │  700604.8 │  755763.3 │  700610.8 │  755763.3 │       6.0 │
│ EMF_bt_access_ductbank_02xC │  700604.8 │ 755946.06 │  700610.8 │ 755946.06 │       6.0 │
│ EMF_bt_access_ductbank_08xC │  700604.8 │  755965.9 │  700610.8 │  755965.9 │       6.0 │
│ EMF_bt_access_ductbank_01xC │  700373.1 │ 756155.94 │  700379.1 │ 756155.94 │ 6.0000024 │
│ EMF_bt_access_ductbank_04xC │ 700372.94 │  756304.5 │ 700378.94 │  756304.5 │  5.999999 │
│ EMF_bt_access_ductbank_03xC │  700610.8 │ 756291.25 │  700604.8 │ 756291.25 │ 

In [9]:
ac_access_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'ac_access' GROUP BY layer ORDER BY layer")
print(ac_access_group)

┌─────────────────────────────┬────────────────────┐
│            layer            │    total_length    │
│           varchar           │       double       │
├─────────────────────────────┼────────────────────┤
│ EMF_bt_access_ductbank_01xC │  6.000002384185791 │
│ EMF_bt_access_ductbank_02xC │                6.0 │
│ EMF_bt_access_ductbank_03xC │                6.0 │
│ EMF_bt_access_ductbank_04xC │ 11.999999046325684 │
│ EMF_bt_access_ductbank_05xC │  12.00098180770874 │
│ EMF_bt_access_ductbank_07xC │                6.0 │
│ EMF_bt_access_ductbank_08xC │                6.0 │
└─────────────────────────────┴────────────────────┘



In [10]:
ac_access_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'ac_access' GROUP BY layer ORDER BY count DESC")
print(ac_access_count)

┌─────────────────────────────┬───────┐
│            layer            │ count │
│           varchar           │ int64 │
├─────────────────────────────┼───────┤
│ EMF_bt_access_ductbank_04xC │     2 │
│ EMF_bt_access_ductbank_05xC │     2 │
│ EMF_bt_access_ductbank_08xC │     1 │
│ EMF_bt_access_ductbank_03xC │     1 │
│ EMF_bt_access_ductbank_02xC │     1 │
│ EMF_bt_access_ductbank_01xC │     1 │
│ EMF_bt_access_ductbank_07xC │     1 │
└─────────────────────────────┴───────┘



QTY. OF DC CABLES (m)
=

In [11]:
print(dc_cables)

┌─────────────────────────┬───────────┬──────────┬───────────┬───────────┬───────────┐
│          layer          │  start_x  │ start_y  │   end_x   │   end_y   │  length   │
│         varchar         │   float   │  float   │   float   │   float   │   float   │
├─────────────────────────┼───────────┼──────────┼───────────┼───────────┼───────────┤
│ EMF_cabo_solar_positivo │  700380.1 │ 756642.0 │ 700436.25 │  756667.1 │  84.86665 │
│ EMF_cabo_solar_positivo │  700380.1 │ 756642.0 │ 700436.25 │  756667.1 │  84.86665 │
│ EMF_cabo_solar_positivo │  700379.5 │ 756653.0 │ 700436.25 │  756667.1 │  74.49165 │
│ EMF_cabo_solar_positivo │  700379.5 │ 756653.0 │ 700436.25 │  756667.1 │  74.49165 │
│ EMF_cabo_solar_positivo │  700379.5 │ 756653.0 │ 700436.25 │  756667.1 │  74.49165 │
│ EMF_cabo_solar_positivo │  700414.6 │ 756653.0 │ 700436.25 │  756667.1 │  39.36665 │
│ EMF_cabo_solar_positivo │  700414.6 │ 756653.0 │ 700436.25 │  756667.1 │  39.36665 │
│ EMF_cabo_solar_positivo │  700414.6 │ 756

In [12]:
dc_cables_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_cables' GROUP BY layer ORDER BY layer")
print(dc_cables_group)

print(f'The total length of DC cables is {round(conn.sql("SELECT SUM(length) FROM 'dc_cables'").fetchone()[0],3)} meters')

┌─────────────────────────┬───────────────────┐
│          layer          │   total_length    │
│         varchar         │      double       │
├─────────────────────────┼───────────────────┤
│ EMF_cabo_solar_negativo │ 413350.1921043396 │
│ EMF_cabo_solar_positivo │ 413350.1921043396 │
└─────────────────────────┴───────────────────┘

The total length of DC cables is 826700.384 meters


QTY. OF DC DUCTS (m and un.)
=

In [13]:
print(dc_ducts)

┌──────────────────────┬───────────┬──────────┬───────────┬───────────┬───────────┐
│        layer         │  start_x  │ start_y  │   end_x   │   end_y   │  length   │
│       varchar        │   float   │  float   │   float   │   float   │   float   │
├──────────────────────┼───────────┼──────────┼───────────┼───────────┼───────────┤
│ EMF_bt_eletroduto_1S │  700571.3 │ 756609.0 │  700570.2 │  756612.1 │      4.25 │
│ EMF_bt_eletroduto_1S │  700624.0 │ 756578.0 │ 700619.75 │ 756538.94 │  40.28141 │
│ EMF_bt_eletroduto_1S │ 700601.56 │ 756576.0 │  700600.4 │  756579.1 │    4.3125 │
│ EMF_bt_eletroduto_1S │  700618.6 │ 756557.0 │ 700619.75 │ 756538.94 │   19.1875 │
│ EMF_bt_eletroduto_1S │  700618.6 │ 756536.0 │ 700619.75 │  756538.7 │    3.8125 │
│ EMF_bt_eletroduto_1S │  700599.1 │ 756521.0 │ 700597.94 │  756524.1 │    4.3125 │
│ EMF_bt_eletroduto_1S │  700599.1 │ 756521.0 │ 700578.44 │  756513.4 │   28.3125 │
│ EMF_bt_eletroduto_1S │  700618.6 │ 756504.0 │ 700619.75 │  756507.2 │    4

In [14]:
print(conn.sql('SELECT * FROM dc_cables ORDER BY length DESC LIMIT 1'))

┌─────────────────────────┬───────────┬──────────┬──────────┬──────────┬──────────┐
│          layer          │  start_x  │ start_y  │  end_x   │  end_y   │  length  │
│         varchar         │   float   │  float   │  float   │  float   │  float   │
├─────────────────────────┼───────────┼──────────┼──────────┼──────────┼──────────┤
│ EMF_cabo_solar_positivo │ 700173.44 │ 756372.0 │ 700285.3 │ 756385.3 │ 276.4375 │
└─────────────────────────┴───────────┴──────────┴──────────┴──────────┴──────────┘



In [15]:
dc_ducts_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_ducts' GROUP BY layer ORDER BY layer")
print(dc_ducts_group)

┌──────────────────────┬───────────────────┐
│        layer         │   total_length    │
│       varchar        │      double       │
├──────────────────────┼───────────────────┤
│ EMF_bt_eletroduto_1S │ 3437.305408000946 │
│ EMF_bt_eletroduto_2S │ 3861.720859527588 │
│ EMF_bt_eletroduto_3S │ 16131.74678659439 │
└──────────────────────┴───────────────────┘



In [16]:
dc_ducts_count = conn.sql("SELECT layer, COUNT(layer) as count FROM 'dc_ducts' GROUP BY layer ORDER BY count DESC")
print(dc_ducts_count)

print(f'The total quantity of DC ducts is {conn.sql("SELECT COUNT(layer) FROM 'dc_ducts'").fetchone()[0]}')

┌──────────────────────┬───────┐
│        layer         │ count │
│       varchar        │ int64 │
├──────────────────────┼───────┤
│ EMF_bt_eletroduto_3S │  1605 │
│ EMF_bt_eletroduto_2S │   225 │
│ EMF_bt_eletroduto_1S │   223 │
└──────────────────────┴───────┘

The total quantity of DC ducts is 2053


QTY. OF DC TRENCHES (m)
=

In [17]:
print(dc_trenches)

┌─────────────┬───────────┬──────────┬───────────┬───────────┬───────────┐
│    layer    │  start_x  │ start_y  │   end_x   │   end_y   │  length   │
│   varchar   │   float   │  float   │   float   │   float   │   float   │
├─────────────┼───────────┼──────────┼───────────┼───────────┼───────────┤
│ EMF_DC_vala │  700579.6 │ 756157.0 │  700591.4 │  756157.0 │     11.75 │
│ EMF_DC_vala │  700365.8 │ 756686.0 │  700365.8 │  756675.0 │      11.0 │
│ EMF_DC_vala │  700401.6 │ 756675.0 │ 700402.75 │  756675.0 │  1.098843 │
│ EMF_DC_vala │ 700402.75 │ 756675.0 │ 700436.25 │  756667.4 │ 36.069645 │
│ EMF_DC_vala │ 700437.44 │ 756664.0 │ 700436.25 │  756664.0 │    1.1875 │
│ EMF_DC_vala │ 700438.06 │ 756653.0 │ 700436.25 │  756653.0 │  1.808325 │
│ EMF_DC_vala │ 700438.06 │ 756642.0 │ 700436.25 │  756667.1 │ 26.933325 │
│ EMF_DC_vala │  700497.3 │ 756642.0 │ 700520.25 │  756634.4 │  25.20309 │
│ EMF_DC_vala │ 700521.44 │ 756631.0 │ 700520.25 │  756631.0 │    1.1875 │
│ EMF_DC_vala │  700522.7

In [18]:
dc_trenches_group = conn.sql("SELECT layer,SUM(length) as total_length FROM 'dc_trenches' GROUP BY layer ORDER BY layer")
print(dc_trenches_group)

print(f'\nThe total length of DC trenches is {round(conn.sql("SELECT SUM(length) FROM 'dc_trenches'").fetchone()[0],3)} meters')

┌─────────────┬──────────────────┐
│    layer    │   total_length   │
│   varchar   │      double      │
├─────────────┼──────────────────┤
│ EMF_DC_vala │ 6388.36786699295 │
└─────────────┴──────────────────┘


The total length of DC trenches is 6388.368 meters


OVERVIEW
=

In [19]:
with pd.ExcelWriter(r"C:\Users\Usuario\Desktop\Reask - BOQ.xlsx") as xlsx:
    ac_cables_group.df().to_excel(xlsx, sheet_name='AC Trenches (m)', index=False)
    ac_access_group.df().to_excel(xlsx, sheet_name='AC Ducts Access (m)', index=False)
    ac_access_count.df().to_excel(xlsx, sheet_name='AC Ducts Access (un.)', index=False)
    dc_cables_group.df().to_excel(xlsx, sheet_name='DC Cables (m)', index=False)
    dc_ducts_group.df().to_excel(xlsx, sheet_name='DC Ducts (m)', index=False)
    dc_ducts_count.df().to_excel(xlsx, sheet_name='DC Ducts (un.)', index=False)
    dc_trenches_group.df().to_excel(xlsx, sheet_name='DC Trenches (un.)', index=False)


conn.close()
